# CIFAR-100 — improve your best model (Colab)

Use this notebook **after** you have a strong **Stage B** checkpoint (`se_resnet_from_34pct/best.pt`, ~65% test). Goal: **push accuracy further** (Stage C–style finetune, new hyperparameters, extra epochs) without re-running the full baseline pipeline.

**Not** for reproducing the first ~65% run — use **`CIFAR_Quick.ipynb`** + `quick_se_resnet_results.sh` for that.

**Setup:** GPU runtime. Same bundle as the quick notebook: upload `artifacts/cifar_hydra_project.tar.gz` to **`My Drive/Colab_CIFAR/`**, unpack below.

**Tip:** set a **unique `RUN_NAME`** per experiment so checkpoints stay separate on Drive.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
import os
import pathlib
import shutil
import sys
import tarfile

WORK_ROOT = pathlib.Path('/content')
PROJECT_ROOT = WORK_ROOT / 'cifar-hydra'
MY_DRIVE = pathlib.Path('/content/drive/MyDrive')
DRIVE_ROOT = MY_DRIVE / 'Colab_CIFAR'
BUNDLE_NAME = 'cifar_hydra_project.tar.gz'

MANUAL_BUNDLE = None


def resolve_bundle_path():
    if MANUAL_BUNDLE is not None and MANUAL_BUNDLE.is_file():
        return MANUAL_BUNDLE
    for p in (DRIVE_ROOT / BUNDLE_NAME, MY_DRIVE / BUNDLE_NAME):
        if p.is_file():
            return p
    if MY_DRIVE.is_dir():
        for p in MY_DRIVE.rglob(BUNDLE_NAME):
            if p.is_file():
                return p
    return None


BUNDLE_PATH = resolve_bundle_path()
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

if BUNDLE_PATH is None or not BUNDLE_PATH.exists():
    raise FileNotFoundError(f'Missing {BUNDLE_NAME}. Upload it to My Drive/Colab_CIFAR/.')

with tarfile.open(BUNDLE_PATH, 'r:gz') as archive:
    if sys.version_info >= (3, 12):
        archive.extractall(PROJECT_ROOT, filter='data')
    else:
        archive.extractall(PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print('Bundle:', BUNDLE_PATH)
print('Project:', PROJECT_ROOT)


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


In [ ]:
import subprocess
import torch

subprocess.run(['nvidia-smi'], check=False)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
import os
import sys
import torch

DATA_DIR = DRIVE_ROOT / 'data'
SAVE_DIR = DRIVE_ROOT / 'checkpoints'
DATA_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

os.environ['PYTHON_BIN'] = sys.executable
os.environ['DATA_DIR'] = str(DATA_DIR)
os.environ['SAVE_DIR'] = str(SAVE_DIR)
os.environ['DEVICE'] = 'cuda' if torch.cuda.is_available() else 'cpu'
os.environ['NUM_WORKERS'] = '2'

print('DATA_DIR =', DATA_DIR)
print('SAVE_DIR =', SAVE_DIR)
print('DEVICE =', os.environ['DEVICE'])


## (Optional) Inspect Stage B checkpoint

Confirm **`best_eval_accuracy` ~0.65** before spending GPU time on finetuning.

In [ ]:
import torch

STAGE_B = SAVE_DIR / 'se_resnet_from_34pct' / 'best.pt'
if not STAGE_B.is_file():
    raise FileNotFoundError(
        f'Missing {STAGE_B}. Copy your recovered best.pt here or run CIFAR_Quick first.'
    )
ck = torch.load(STAGE_B, map_location='cpu', weights_only=False)
print('epoch:', ck.get('epoch'), 'best_eval_accuracy:', ck.get('best_eval_accuracy'))


## SOTA experiment — Stage C–style finetune

Runs **`scripts/iterate_from_stageb_stage_c_mixema.sh`** (RandAugment + Mixup/CutMix + EMA, low LR). **Edit the variables** in the next cell (`RUN_NAME`, `EPOCHS`, `LR`, …) between runs.

For deeper changes (architecture, loss, etc.), call **`train.py`** with the same flags as in that script — see `python train.py --help` in a small cell if needed.

In [ ]:
import os
import subprocess
import sys

# --- Edit for each experiment ---
RUN_NAME = 'sota_se_resnet_exp1'
EPOCHS = '40'
LR = '0.01'
WARMUP_EPOCHS = '4'
BATCH_SIZE = '128'

STAGE_B_PT = SAVE_DIR / 'se_resnet_from_34pct' / 'best.pt'
if not STAGE_B_PT.is_file():
    raise FileNotFoundError(STAGE_B_PT)

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['TQDM_DISABLE'] = '1'
env['PYTHON_BIN'] = sys.executable
env['DATA_DIR'] = str(DATA_DIR)
env['SAVE_DIR'] = str(SAVE_DIR)
env['DEVICE'] = os.environ['DEVICE']
env['NUM_WORKERS'] = os.environ.get('NUM_WORKERS', '2')
env['BATCH_SIZE'] = BATCH_SIZE
env['INIT_CKPT'] = str(STAGE_B_PT)
env['RUN_NAME'] = RUN_NAME
env['EPOCHS'] = EPOCHS
env['LR'] = LR
env['WARMUP_EPOCHS'] = WARMUP_EPOCHS

process = subprocess.Popen(
    ['bash', 'scripts/iterate_from_stageb_stage_c_mixema.sh'],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end='')
    sys.stdout.flush()
rc = process.wait()
if rc != 0:
    raise subprocess.CalledProcessError(rc, ['bash', 'scripts/iterate_from_stageb_stage_c_mixema.sh'])


## Official test on the new run

Point **`CKPT`** at **`SAVE_DIR / RUN_NAME / best.pt`** (match `RUN_NAME` above).

In [ ]:
import os
import re
import subprocess
import sys

RUN_NAME = 'sota_se_resnet_exp1'
CKPT = SAVE_DIR / RUN_NAME / 'best.pt'
if not CKPT.is_file():
    raise FileNotFoundError(CKPT)

env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
completed = subprocess.run(
    [
        sys.executable,
        'results.py',
        '--checkpoint',
        str(CKPT),
        '--data-dir',
        str(DATA_DIR),
        '--batch-size',
        '256',
        '--num-workers',
        os.environ.get('NUM_WORKERS', '2'),
        '--device',
        os.environ['DEVICE'],
    ],
    check=True,
    capture_output=True,
    text=True,
    env=env,
)
print(completed.stdout, end='')
m = re.search(r'Test accuracy:\s*([\d.]+)', completed.stdout)
if m:
    acc = float(m.group(1))
    print(f'>>> Official test accuracy: {acc:.4f} ({100 * acc:.2f}%)')


## Ideas to try next

- **Longer / cooler schedule:** more `EPOCHS`, lower `LR`, longer `WARMUP_EPOCHS`.
- **Stronger EMA:** tweak `--ema-decay` in `iterate_from_stageb_stage_c_mixema.sh` (copy to a custom script if needed).
- **Second pass:** init from your **best SOTA run** instead of Stage B (`INIT_CKPT` → previous `sota_* / best.pt`).
- **Local tuning:** `bash scripts/local_runner.sh stage-c` with the same env vars on a Mac (MPS) for cheap iteration.

CIFAR-100 “true” SOTA is much higher than ~65%; this project targets **your** pipeline’s best checkpoint.